# 03 生成最终五列三状态信号

本 Notebook 重新读取唯一现货输入，分别重建负向和正向候选池，在各自的 Development+Validation 上冻结最终 Top1，然后输出一个严格五列的 CSV。

`date` 是形成日收盘后的下一实际现货交易日，也就是开盘执行日；`three_state` 是该形成日计算出的原始三状态。退出信号在该 `date` 生效，只在对应退出信号发生的实际执行日将最终状态置为 `0`；下一交易日恢复原始三状态。Test 只在冻结后观察，不参与这份 CSV 的生成选择。

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 60)

PACKAGE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src" / "pool_runner.py").is_file()
)
sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from signal_export import SIGNAL_COLUMNS, build_frozen_signal_export

OUTPUT_PATH = Path(
    os.environ.get(
        "1545_SIGNAL_CSV_PATH",
        str(PACKAGE_ROOT / "IC_1545_frozen_exit_signals.csv"),
    )
).expanduser()

def progress(message: str) -> None:
    print(f"[1545-SIGNAL] {message}", flush=True)

signal_df, metadata, pools = build_frozen_signal_export(progress=progress)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
signal_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

assert list(signal_df.columns) == list(SIGNAL_COLUMNS)
assert signal_df["date"].is_unique
assert signal_df["three_state"].isin([-1, 0, 1]).all()
assert signal_df["final_three_state"].isin([-1, 0, 1]).all()
assert signal_df["minus_exit_signal"].isin([0, 1]).all()
assert signal_df["plus_exit_signal"].isin([0, 1]).all()

print("\n输入与输出：")
print("spot:", metadata["input_manifest"]["spot"]["path"])
print("rows:", len(signal_df))
print("date:", signal_df["date"].min(), "->", signal_df["date"].max())
print("CSV:", OUTPUT_PATH.resolve())
print("columns:", list(signal_df.columns))

## 两侧最终冻结参数

In [ ]:
freeze_rows = []
for side in ("minus", "plus"):
    item = metadata["freeze"][side]
    freeze_rows.append({
        "side": side,
        "version": item.get("version"),
        "action": item.get("action"),
        "core_logic_name": item.get("core_logic_name"),
        "candidate_id": item.get("candidate_id"),
        "score_variant": item.get("score_variant"),
        "threshold_quantile": item.get("threshold_quantile"),
        "threshold_value": item.get("threshold_value"),
        "min_state_age": item.get("min_state_age"),
        "confirm_days": item.get("confirm_days"),
        "cooldown_days": item.get("cooldown_days"),
        "selection_data_end": item.get("selection_data_end"),
        "test_used_for_selection": item.get("test_used_for_selection"),
    })
display(pd.DataFrame(freeze_rows).round(6))

## 五列 CSV 预览与退出信号明细

In [ ]:
print("前 10 行：")
display(signal_df.head(10))
print("最后 10 行：")
display(signal_df.tail(10))

print("退出信号数量：")
display(pd.DataFrame([{
    "minus_exit_days": int(signal_df["minus_exit_signal"].sum()),
    "plus_exit_days": int(signal_df["plus_exit_signal"].sum()),
    "all_exit_days": int(
        ((signal_df["minus_exit_signal"] == 1) |
         (signal_df["plus_exit_signal"] == 1)).sum()
    ),
    "original_state_counts": signal_df["three_state"].value_counts().sort_index().to_dict(),
    "final_state_counts": signal_df["final_three_state"].value_counts().sort_index().to_dict(),
}]))

exit_rows = signal_df.loc[
    (signal_df["minus_exit_signal"] == 1) |
    (signal_df["plus_exit_signal"] == 1)
].copy()
if exit_rows.empty:
    print("当前数据没有冻结退出信号。")
else:
    exit_rows.insert(
        2,
        "exit_side",
        np.where(
            exit_rows["minus_exit_signal"].eq(1),
            "-1→0",
            "1→0",
        ),
    )
    print("全部冻结退出日（date 为实际开盘执行日）：")
    display(exit_rows)

print("\n已写出严格五列 CSV：", OUTPUT_PATH.resolve())
print("FINAL_SIGNAL_EXPORT_END")